# Evaluate a model on a split

`orinth.models.predictor()` returns the object `/inference` runs, and
`orinth.models.parameters()` gives it the same defaults the studio sends. So a
number computed here is a number the platform will reproduce — which is the
only reason to compute it here rather than in a script.

In [ ]:
import orinth

for model in orinth.models.list():
    print(f"{model.id:32} {model.family:22} {model.source:10} available={model.available}")

MODEL_ID = next(model.id for model in orinth.models.list() if model.available)
DATASET_ID = next(
    dataset.id
    for dataset in orinth.datasets.list()
    if dataset.task_type in {"classification", "object_detection", "segmentation"}
    and dataset.splits.get("valid")
)
SPLIT = "valid"
print("\nscoring", MODEL_ID, "on", DATASET_ID, SPLIT)

## Predict

The first call loads weights and is slow; the predictor is cached after that, so
re-running this cell costs only inference. `image_label()` collapses a detection
list to one verdict the way the studio does — largest mask wins, nothing means
`Normal` — instead of restating that rule differently here.

In [ ]:
from pathlib import Path

predictor = orinth.models.predictor(MODEL_ID)
params = orinth.models.parameters(confidence_threshold=0.65, iou_threshold=0.7)

frame = orinth.datasets.load(DATASET_ID, SPLIT, limit=200)
rows = []
for row in frame.iter_rows(named=True):
    path = Path(row["path"])
    scores = predictor.classify(path, params)
    if scores:
        predicted = max(scores, key=scores.get)
        confidence = float(scores[predicted])
    else:
        detections = predictor.predict(path, params)
        predicted = orinth.models.image_label(detections)
        confidence = max((float(d.confidence) for d in detections), default=0.0)
    rows.append(
        {
            "item": row["item_id"],
            "truth": row["label"] or "Normal",
            "predicted": predicted,
            "confidence": confidence,
        }
    )

print(len(rows), "items scored")

## Confusion matrix

Read the off-diagonal first. Overall accuracy hides the failure that matters —
a model that never predicts the rare class scores well and is useless.

In [ ]:
import polars as pl

scored = pl.DataFrame(rows)
accuracy = (scored["truth"] == scored["predicted"]).mean()
print(f"accuracy {accuracy:.3f} over {scored.height} items\n")

scored.pivot(on="predicted", index="truth", values="item", aggregate_function="len").fill_null(0)

## Per class

Precision is "when it said this, was it right"; recall is "of the ones that were
this, how many did it find". A class with high precision and low recall is a
threshold problem; low on both is a data problem.

In [ ]:
report = []
for label in sorted(set(scored["truth"].to_list()) | set(scored["predicted"].to_list())):
    predicted = scored.filter(pl.col("predicted") == label).height
    actual = scored.filter(pl.col("truth") == label).height
    hit = scored.filter((pl.col("truth") == label) & (pl.col("predicted") == label)).height
    precision = hit / predicted if predicted else 0.0
    recall = hit / actual if actual else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    report.append(
        {"label": label, "support": actual, "precision": precision, "recall": recall, "f1": f1}
    )

pl.DataFrame(report).sort("f1")

## Keep the score

Logged as a run, the evaluation sits beside the training run that produced the
model — which is what makes "did the retrain help" answerable.

In [ ]:
with orinth.runs.start(
    f"eval {MODEL_ID}", params={"model": MODEL_ID, "split": SPLIT}, dataset_id=DATASET_ID
) as run:
    run.log(accuracy=float(accuracy))
    for entry in report:
        run.log(**{f"f1_{entry['label']}": float(entry["f1"])})
    run.log_artifact(scored.write_csv().encode("utf-8"), name="predictions.csv")